# Análise do Hiperparâmetro de Regularização ($C$) em LinearSVC e Regressão Logística

**Pergunta central:** O classificador com menor $C$ (mais regularizado) valida melhor do que o $C=1$ atual? Reduzir a complexidade do modelo diminui o sobreajuste (*overfitting*) sem prejudicar a capacidade de generalização?

Para garantir comparabilidade rigorosa com os notebooks `02_LR_training`, `03_SVM_training` e `05_feature_dim_analysis`:
- **Mesmos dados:** `dados_preparados.pkl`, utilizando exclusivamente a partição de treino (`X_tr`, `y_tr`). O conjunto de teste (`X_te`, `y_te`) permanece estritamente isolado.
- **Mesmos transformers:** `transformers.pkl`, configuração `word+char` (unigramas e bigramas de palavras com `min_df=2` + 3- a 5-gramas de caracteres com `min_df=3`), totalizando ~114 mil features esparsas.
- **Mesmos folds de validação cruzada:** `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`.
- **Classificadores avaliados:** `LinearSVC` e `LogisticRegression` com balanceamento de classes (`class_weight="balanced"`).
- **Grade de exploração de $C$:** $\{0.01, 0.03, 0.1, 0.3, 1, 3, 10\}$.

In [ ]:
# Resolve input and output paths from the supervised research root.
from pathlib import Path
import sys

_cwd = Path.cwd().resolve()
_candidates = [candidate for parent in (_cwd, *_cwd.parents)
               for candidate in (parent, parent / "machine-learning/supervised-learning")]
ROOT = next((candidate for candidate in _candidates
             if (candidate / "modelo_olimpo.py").is_file()
             and (candidate / "history/baselines/01_Data_Prep.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the Olimpo repository.")
DATA = ROOT / "data"
HISTORY_RESULTS = ROOT / "history/resultados"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Original experiment starts here.
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_validate

# 1. Carregamento dos dados e pipelines de transformação
X_tr, X_te, y_tr, y_te = joblib.load((DATA / 'dados_preparados.pkl'))
configs = joblib.load((DATA / 'transformers.pkl'))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Instâncias de treino: {len(X_tr):,}")
print("Configuração utilizada: word+char")

## Validação Cruzada: Varredura de $C$

Avaliamos o desempenho com a métrica **F1-Macro** (média harmônica entre precisão e revocação para ambas as classes). Calculamos:
- $F1_{\text{treino}}$: Desempenho no subconjunto de treino de cada fold.
- $F1_{\text{CV}}$: Desempenho na validação cruzada (média dos 5 folds).
- Desvio padrão ($\sigma_{\text{CV}}$): Estabilidade entre os folds.
- Gap de generalização: $F1_{\text{treino}} - F1_{\text{CV}}$ (indicador direto de overfitting).

Os resultados são salvos em `../resultados/resultados_C.csv`. Caso o arquivo já exista, ele é carregado para visualização rápida.

In [ ]:
modelos = {
    "SVC": LinearSVC(class_weight="balanced", random_state=42, max_iter=5000),
    "LR":  LogisticRegression(max_iter=2000, class_weight="balanced"),
}

valores_C = [0.01, 0.03, 0.1, 0.3, 1, 3, 10]
csv_path = (HISTORY_RESULTS / 'resultados_C.csv')

if os.path.exists(csv_path):
    print(f"Carregando resultados pré-calculados de {csv_path}...")
    res = pd.read_csv(csv_path)
else:
    print("Executando validação cruzada para todos os cenários...")
    linhas = []
    for nome, clf in modelos.items():
        for C in valores_C:
            pipe = Pipeline([
                ("prep", ColumnTransformer(configs["word+char"])),
                ("clf", clone(clf).set_params(C=C))
            ])
            r = cross_validate(pipe, X_tr, y_tr, cv=cv, scoring="f1_macro", return_train_score=True, n_jobs=-1)
            linhas.append({
                "modelo": nome,
                "C": C,
                "f1_treino": r["train_score"].mean(),
                "f1_cv": r["test_score"].mean(),
                "f1_cv_std": r["test_score"].std(),
                "gap": r["train_score"].mean() - r["test_score"].mean(),
                **{f"fold{i+1}": s for i, s in enumerate(r["test_score"])}
            })
            print(f"{linhas[-1]['modelo']} | C={C:<5} | treino={linhas[-1]['f1_treino']:.4f} | cv={linhas[-1]['f1_cv']:.4f} ± {linhas[-1]['f1_cv_std']:.4f} | gap={linhas[-1]['gap']:.4f}", flush=True)
    res = pd.DataFrame(linhas)
    res.to_csv(csv_path, index=False)

## Tabelas Comparativas de Resultados

In [ ]:
colunas_principais = ["modelo", "C", "f1_treino", "f1_cv", "f1_cv_std", "gap"]

print("=== LinearSVC (Variação de C) ===")
display(res[res.modelo == "SVC"][colunas_principais].round(4).reset_index(drop=True))

print("\n=== Regressão Logística (Variação de C) ===")
display(res[res.modelo == "LR"][colunas_principais].round(4).reset_index(drop=True))

## Curvas de Desempenho e Gap de Generalização

Os gráficos abaixo ilustram o comportamento de $F1_{\text{treino}}$, $F1_{\text{CV}}$ e o Gap ($F1_{\text{treino}} - F1_{\text{CV}}$) à medida que o parâmetro $C$ varia na escala logarítmica.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

cores = {"SVC": "#2a78d6", "LR": "#eb6834"}

for ax, modelo in zip(axes, ["SVC", "LR"]):
    sub = res[res.modelo == modelo].sort_values("C")
    
    ax.plot(sub["C"], sub["f1_treino"], marker="o", lw=2, label="F1 Treino", color="#555555", linestyle="--")
    ax.plot(sub["C"], sub["f1_cv"], marker="s", lw=2, label="F1 CV", color=cores[modelo])
    ax.fill_between(sub["C"], sub["f1_cv"] - sub["f1_cv_std"], sub["f1_cv"] + sub["f1_cv_std"], alpha=0.15, color=cores[modelo])
    
    # Destacar o ponto C=1 (configuração original)
    ponto_base = sub[sub.C == 1.0].iloc[0]
    ax.plot(ponto_base.C, ponto_base.f1_cv, marker="*", ms=16, color="none", mec="#222222", mew=2, label="Base (C=1)")
    
    ax.set_xscale("log")
    ax.set_title(f"{modelo}: F1 Treino vs F1 CV", fontsize=12)
    ax.set_xlabel("Parâmetro C (escala log)", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

axes[0].set_ylabel("F1 Macro", fontsize=10)
plt.tight_layout()
plt.show()

# Gráfico de evolução do Gap
fig, ax = plt.subplots(figsize=(8, 4.5))
for modelo in ["SVC", "LR"]:
    sub = res[res.modelo == modelo].sort_values("C")
    ax.plot(sub["C"], sub["gap"], marker="^", lw=2, ms=7, label=f"Gap {modelo}", color=cores[modelo])

ax.set_xscale("log")
ax.set_title("Evolução do Gap de Generalização ($F1_{treino} - F1_{CV}$)", fontsize=12)
ax.set_xlabel("Parâmetro C (escala log)", fontsize=10)
ax.set_ylabel("Gap", fontsize=10)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Conclusões

1. **LinearSVC:**
   - Com $C \le 0.03$, o modelo sofre de subajuste (*underfitting*), com o F1 da CV caindo de 0.9295 para 0.8802 ($C=0.01$).
   - O melhor F1 de validação cruzada ocorre exatamente em **$C=1.0$** ($0.9295 \pm 0.0099$).
   - Valores maiores ($C=3$ e $C=10$) saturam o treino em 1.0 sem produzir ganho na CV ($0.9283$).
   - **Conclusão:** O valor padrão **$C=1.0$** é o ponto ótimo de operação do LinearSVC.

2. **Regressão Logística (LR):**
   - Para valores baixos de $C$, a penalização $L_2$ é muito agressiva, resultando em F1 CV baixo ($0.8417$ em $C=0.01$).
   - O desempenho sobe monotonicamente até atingir o ápice em **$C=10.0$** ($0.9281 \pm 0.0109$), praticamente empatando com o LinearSVC ($0.9295$).
   - **Conclusão:** Em dados de alta dimensionalidade esparsa, a Regressão Logística se beneficia de regularização mais fraca (maior $C$) do que o padrão $C=1$.